# MercadoLibre | Data Exploration & Quality Assessment

## Project Overview

This project analyzes user behavior across the purchase funnel and customer retention over time.

The analysis uses SQL to explore event data, measure conversion between funnel stages, and evaluate retention by user cohorts.

## Data Sources

- `mercadolibre_funnel`: User events throughout the purchase journey.
- `mercadolibre_retention`: User activity after registration.

Both tables are stored in a local SQLite database.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

# Database location
db_path = Path("../data/raw/tripleten.db").resolve()

# Connect in read-only mode
connection = sqlite3.connect(
    db_path.as_uri() + "?mode=ro",
    uri=True
)

# Verify the connection
query = "SELECT name FROM sqlite_master WHERE type = 'table' AND name LIKE 'mercadolibre_%'"

tables = pd.read_sql_query(query, connection)

print("Database connection successful!")
display(tables)

Database connection successful!


,name
0,mercadolibre_funnel
1,mercadolibre_retention


## 1. Initial Data Exploration

### 1.1 Dataset Size and Unique Users

The first step is to examine the number of records and unique users in each dataset.

In [2]:
# Dataset size and unique users

query = """
SELECT
    'Funnel' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users
FROM mercadolibre_funnel

UNION ALL

SELECT
    'Retention' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users
FROM mercadolibre_retention;
"""

dataset_summary = pd.read_sql_query(query, connection)

display(dataset_summary)

,dataset,total_rows,unique_users
0,Funnel,61235,2901
1,Retention,696000,12000


### 1.2 Date Coverage

This step checks the earliest and latest dates available in each dataset to define the analysis periods.

In [3]:
# Check date coverage using SQL

query = """
SELECT
    'Funnel' AS dataset,
    MIN(event_date) AS start_date,
    MAX(event_date) AS end_date
FROM mercadolibre_funnel

UNION ALL

SELECT
    'Retention' AS dataset,
    MIN(signup_date) AS start_date,
    MAX(signup_date) AS end_date
FROM mercadolibre_retention;
"""

date_coverage = pd.read_sql_query(query, connection)

display(date_coverage)

,dataset,start_date,end_date
0,Funnel,2024-01-10 15:45:15,2025-12-31 8:03:14
1,Retention,2025-01-01,2025-08-25


### 1.3 Event Distribution

This step identifies the available funnel events and their frequency in the dataset.

In [4]:
# Event distribution

query = """
SELECT
    event_name,
    COUNT(*) AS total_events,
    COUNT(DISTINCT user_id) AS unique_users
FROM mercadolibre_funnel
GROUP BY event_name
ORDER BY total_events DESC;
"""

event_distribution = pd.read_sql_query(query, connection)

display(event_distribution)

,event_name,total_events,unique_users
0,page_view,27372,2901
1,user_engagement,6837,2901
2,session_start,6837,2901
3,view_item,4113,2376
4,select_item,4113,2376
5,scroll,4053,2359
6,first_visit,2752,2752
7,view_search_results,1693,1328
8,click,1343,1109
9,view_promotion,1033,893


### 1.4 Geographic Distribution

This step examines the countries represented in each dataset and the number of unique users associated with them.

In [5]:
# Geographic distribution by dataset

query = """
SELECT
    'Funnel' AS dataset,
    country,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users
FROM mercadolibre_funnel
GROUP BY country

UNION ALL

SELECT
    'Retention' AS dataset,
    country,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users
FROM mercadolibre_retention
GROUP BY country

ORDER BY dataset, unique_users DESC;
"""

country_distribution = pd.read_sql_query(query, connection)

display(country_distribution)

,dataset,country,total_rows,unique_users
0,Funnel,Brazil,15703,728
1,Funnel,Mexico,11886,588
2,Funnel,Colombia,8810,419
3,Funnel,Argentina,7944,379
4,Funnel,Peru,5228,235
5,Funnel,Chile,5099,233
6,Funnel,Ecuador,2627,137
7,Funnel,Bolivia,1643,75
8,Funnel,Paraguay,1116,54
9,Funnel,Uruguay,1179,53


## 2. Data Quality Assessment

### 2.1 Missing Values

This step checks for missing values in the key fields used for funnel and retention analysis.

In [6]:
# Check missing values in key columns

query = """
SELECT
    'Funnel' AS dataset,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS missing_user_id,
    SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END) AS missing_session_id,
    SUM(CASE WHEN event_name IS NULL THEN 1 ELSE 0 END) AS missing_event_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END) AS missing_date,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS missing_country
FROM mercadolibre_funnel

UNION ALL

SELECT
    'Retention' AS dataset,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END),
    NULL,
    NULL,
    SUM(CASE WHEN signup_date IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END)
FROM mercadolibre_retention;
"""

missing_values = pd.read_sql_query(query, connection)

display(missing_values)

,dataset,total_rows,missing_user_id,missing_session_id,missing_event_name,missing_date,missing_country
0,Funnel,61235,0,0.0,0.0,0,0
1,Retention,696000,0,NaN,NaN,0,0


### 2.2 Duplicate Records

This step checks for fully duplicated rows in both datasets. Duplicate records will be reviewed before deciding whether any should be removed.

In [7]:
# Check exact duplicate records

query = """
SELECT
    'Funnel' AS dataset,
    COUNT(*) - (
        SELECT COUNT(*)
        FROM (SELECT DISTINCT * FROM mercadolibre_funnel)
    ) AS duplicate_rows
FROM mercadolibre_funnel

UNION ALL

SELECT
    'Retention' AS dataset,
    COUNT(*) - (
        SELECT COUNT(*)
        FROM (SELECT DISTINCT * FROM mercadolibre_retention)
    ) AS duplicate_rows
FROM mercadolibre_retention;
"""

duplicate_summary = pd.read_sql_query(query, connection)

display(duplicate_summary)

,dataset,duplicate_rows
0,Funnel,6
1,Retention,348000


### 2.3 Duplicate Pattern Investigation

The retention dataset contains 348,000 exact duplicate rows.

This step checks how many times each unique record appears before deciding how to handle duplicates.

In [8]:
# Investigate duplicate frequency in retention data

query = """
WITH record_frequency AS (
    SELECT
        user_id,
        signup_datetime,
        country,
        device_category,
        platform,
        activity_date,
        prob_active,
        day_after_signup,
        active,
        signup_date,
        COUNT(*) AS occurrences
    FROM mercadolibre_retention
    GROUP BY
        user_id,
        signup_datetime,
        country,
        device_category,
        platform,
        activity_date,
        prob_active,
        day_after_signup,
        active,
        signup_date
)

SELECT
    occurrences,
    COUNT(*) AS unique_records,
    SUM(occurrences) AS total_rows
FROM record_frequency
GROUP BY occurrences
ORDER BY occurrences;
"""

duplicate_pattern = pd.read_sql_query(query, connection)

display(duplicate_pattern)

,occurrences,unique_records,total_rows
0,2,348000,696000


### 2.4 Retention Data Deduplication

The retention dataset contains 696,000 rows, but only 348,000 distinct records. Each record appears exactly twice.

For retention analysis, we will use distinct records to avoid double-counting user activity.

The original database will remain unchanged.

In [9]:
# Validate retention data after removing exact duplicates

query = """
WITH retention_clean AS (
    SELECT DISTINCT *
    FROM mercadolibre_retention
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users,
    MIN(signup_date) AS first_signup,
    MAX(signup_date) AS last_signup
FROM retention_clean;
"""

retention_clean_summary = pd.read_sql_query(query, connection)

display(retention_clean_summary)

,total_rows,unique_users,first_signup,last_signup
0,348000,12000,2025-01-01,2025-08-25


### 2.5 Temporal Consistency

This step checks whether activity dates and days since signup are consistent with each user's registration date.

In [10]:
# Validate temporal consistency in retention data

query = """
WITH retention_clean AS (
    SELECT DISTINCT *
    FROM mercadolibre_retention
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN date(activity_date) < date(signup_date)
            THEN 1 ELSE 0
        END
    ) AS activity_before_signup,

    SUM(
        CASE
            WHEN day_after_signup < 0
            THEN 1 ELSE 0
        END
    ) AS negative_days,

    SUM(
        CASE
            WHEN day_after_signup !=
                 CAST(julianday(activity_date) - julianday(signup_date) AS INTEGER)
            THEN 1 ELSE 0
        END
    ) AS inconsistent_day_difference

FROM retention_clean;
"""

temporal_quality = pd.read_sql_query(query, connection)

display(temporal_quality)

,total_rows,activity_before_signup,negative_days,inconsistent_day_difference
0,348000,0,0,0


### 2.6 Empty Identifiers and Event Fields

This step checks for empty or whitespace-only values in the identifiers and event fields required for the analysis.

In [11]:
# Check empty or whitespace-only values in key fields

query = """
SELECT
    'Funnel' AS dataset,
    SUM(CASE WHEN TRIM(user_id) = '' THEN 1 ELSE 0 END) AS empty_user_id,
    SUM(CASE WHEN TRIM(session_id) = '' THEN 1 ELSE 0 END) AS empty_session_id,
    SUM(CASE WHEN TRIM(event_name) = '' THEN 1 ELSE 0 END) AS empty_event_name,
    SUM(CASE WHEN TRIM(country) = '' THEN 1 ELSE 0 END) AS empty_country
FROM mercadolibre_funnel

UNION ALL

SELECT
    'Retention' AS dataset,
    SUM(CASE WHEN TRIM(user_id) = '' THEN 1 ELSE 0 END),
    NULL,
    NULL,
    SUM(CASE WHEN TRIM(country) = '' THEN 1 ELSE 0 END)
FROM mercadolibre_retention;
"""

empty_values = pd.read_sql_query(query, connection)

display(empty_values)

,dataset,empty_user_id,empty_session_id,empty_event_name,empty_country
0,Funnel,0,0.0,0.0,0
1,Retention,0,NaN,NaN,0


### 2.7 Funnel Timestamp Validation

This step examines the event timestamps and compares them with the recorded event dates to identify the correct time unit and detect possible inconsistencies.

In [12]:
# Inspect timestamp values and their corresponding dates

query = """
SELECT
    event_times,
    event_date,
    datetime(event_times, 'unixepoch') AS interpreted_seconds,
    datetime(event_times / 1000.0, 'unixepoch') AS interpreted_milliseconds,
    datetime(event_times / 1000000.0, 'unixepoch') AS interpreted_microseconds
FROM mercadolibre_funnel
WHERE event_times IS NOT NULL
ORDER BY event_date
LIMIT 10;
"""

timestamp_check = pd.read_sql_query(query, connection)

display(timestamp_check)

,event_times,event_date,interpreted_seconds,interpreted_milliseconds,interpreted_microseconds
0,1.704900e+15,2024-01-10 15:45:15,None,None,2024-01-10 15:20:00
1,1.704900e+15,2024-01-10 15:45:24,None,None,2024-01-10 15:20:00
2,1.704900e+15,2024-01-10 15:47:08,None,None,2024-01-10 15:20:00
3,1.704900e+15,2024-01-10 15:47:23,None,None,2024-01-10 15:20:00
4,1.704900e+15,2024-01-10 15:47:28,None,None,2024-01-10 15:20:00
5,1.704870e+15,2024-01-10 5:37:30,None,None,2024-01-10 07:00:00
6,1.704870e+15,2024-01-10 5:37:32,None,None,2024-01-10 07:00:00
7,1.704870e+15,2024-01-10 5:38:06,None,None,2024-01-10 07:00:00
8,1.704870e+15,2024-01-10 5:38:59,None,None,2024-01-10 07:00:00
9,1.704870e+15,2024-01-10 5:39:44,None,None,2024-01-10 07:00:00


### 2.8 Timestamp Consistency

The `event_times` field uses Unix microseconds, but its converted values do not always match `event_date`.

This step measures the differences between both timestamps to determine which field is more appropriate for chronological funnel analysis.

In [13]:
# Compare event timestamps across the full dataset

query = """
WITH timestamp_comparison AS (
    SELECT
        event_date,
        datetime(event_times / 1000000.0, 'unixepoch') AS converted_timestamp,
        ABS(
            (julianday(event_date) -
             julianday(event_times / 1000000.0, 'unixepoch')) * 86400
        ) AS difference_seconds
    FROM mercadolibre_funnel
    WHERE event_times IS NOT NULL
      AND event_date IS NOT NULL
)

SELECT
    COUNT(*) AS total_events,
    ROUND(MIN(difference_seconds), 2) AS min_difference_seconds,
    ROUND(AVG(difference_seconds), 2) AS avg_difference_seconds,
    ROUND(MAX(difference_seconds), 2) AS max_difference_seconds,

    SUM(
        CASE WHEN difference_seconds > 60 THEN 1 ELSE 0 END
    ) AS differences_over_1_minute,

    SUM(
        CASE WHEN difference_seconds > 3600 THEN 1 ELSE 0 END
    ) AS differences_over_1_hour

FROM timestamp_comparison;
"""

timestamp_consistency = pd.read_sql_query(query, connection)

display(timestamp_consistency)

,total_events,min_difference_seconds,avg_difference_seconds,max_difference_seconds,differences_over_1_minute,differences_over_1_hour
0,61235,1.0,2502.76,5000.0,35118,10081


### 2.9 Timestamp Precision Assessment

This step examines the precision of `event_times` and compares the number of distinct timestamps with `event_date`.

The goal is to identify potential limitations before defining the chronological order of funnel events.

In [14]:
# Evaluate timestamp precision

query = """
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT event_times) AS unique_event_times,
    COUNT(DISTINCT event_date) AS unique_event_dates,

    SUM(
        CASE
            WHEN CAST(event_times AS INTEGER) % 1000000000 = 0
            THEN 1 ELSE 0
        END
    ) AS rounded_timestamps

FROM mercadolibre_funnel;
"""

timestamp_precision = pd.read_sql_query(query, connection)

display(timestamp_precision)

,total_events,unique_event_times,unique_event_dates,rounded_timestamps
0,61235,3976,61085,61235


### 2.10 Funnel Event Date Validation

This step checks whether event dates are valid and identifies events sharing the same timestamp within a user session.

These checks help assess whether `event_date` can be used to reconstruct the chronological purchase journey.

In [15]:
# Validate event dates and timestamp ties

query = """
WITH timestamp_groups AS (
    SELECT
        user_id,
        session_id,
        event_date,
        COUNT(*) AS occurrences
    FROM mercadolibre_funnel
    GROUP BY user_id, session_id, event_date
)

SELECT
    (SELECT COUNT(*)
     FROM mercadolibre_funnel) AS total_events,

    (SELECT COUNT(*)
     FROM mercadolibre_funnel
     WHERE datetime(event_date) IS NULL) AS invalid_event_dates,

    (SELECT COUNT(*)
     FROM timestamp_groups
     WHERE occurrences > 1) AS groups_with_same_timestamp,

    (SELECT COALESCE(SUM(occurrences), 0)
     FROM timestamp_groups
     WHERE occurrences > 1) AS events_in_tied_groups;
"""

event_date_validation = pd.read_sql_query(query, connection)

display(event_date_validation)

,total_events,invalid_event_dates,groups_with_same_timestamp,events_in_tied_groups
0,61235,25684,107,214


In [16]:
# Inspect event dates that SQLite cannot parse

query = """
SELECT
    event_date,
    event_name,
    user_id,
    session_id
FROM mercadolibre_funnel
WHERE datetime(event_date) IS NULL
LIMIT 15;
"""

invalid_dates_sample = pd.read_sql_query(query, connection)

display(invalid_dates_sample)

,event_date,event_name,user_id,session_id
0,2025-07-04 0:48:47,user_engagement,ebbdc8e6-d0ec-4dd8-bf98-c3a9ec5c2c68,sid_5815
1,2025-09-11 0:48:23,view_promotion,d2b2dadf-2310-48ab-9e81-e3856b53faa2,sid_990
2,2025-10-26 9:08:49,click,4406b610-4adb-4a02-b3cd-a0bfd2258efc,sid_6202
3,2024-10-27 0:35:40,page_view,86f2a4c7-670b-4422-85e3-3348e4c08891,sid_952
4,2025-04-04 7:53:50,page_view,8749e2bd-2244-4453-a8d4-1c4459fc6d91,sid_3957
5,2024-07-24 9:27:37,view_promotion,208eb7d6-101f-466e-95c7-dc1792b637fc,sid_1020
6,2025-02-21 5:51:13,session_start,41892a3b-b9eb-4c69-bcb2-21d4850263aa,sid_3422
7,2025-09-17 3:06:17,page_view,f67f4ecd-cd0b-4e8f-9ebf-a733defccb01,sid_2284
8,2025-01-13 3:46:04,view_item,0154be3b-361c-4352-a420-4f9a6cdda58f,sid_4576
9,2025-10-20 1:01:43,page_view,a36a7f26-26f4-46f3-8a53-62b22fed7dc8,sid_908


### 2.12 Event Date Standardization Check

Some event dates use single-digit hours, which SQLite does not parse correctly.

This step checks whether standardizing the hour format resolves the issue without changing the original data.

In [17]:
# Validate standardized event dates

query = """
WITH standardized_dates AS (
    SELECT
        event_date,

        CASE
            WHEN substr(event_date, 12, 2) LIKE '%:'
            THEN substr(event_date, 1, 11) || '0' ||
                 substr(event_date, 12)
            ELSE event_date
        END AS standardized_date

    FROM mercadolibre_funnel
)

SELECT
    COUNT(*) AS total_events,

    SUM(
        CASE WHEN datetime(event_date) IS NULL
        THEN 1 ELSE 0 END
    ) AS invalid_before,

    SUM(
        CASE WHEN datetime(standardized_date) IS NULL
        THEN 1 ELSE 0 END
    ) AS invalid_after

FROM standardized_dates;
"""

date_standardization = pd.read_sql_query(query, connection)

display(date_standardization)

,total_events,invalid_before,invalid_after
0,61235,25684,0


### 2.13 Retention Observation Window

This step identifies the available activity period and the number of days observed after user registration.

These results will help define the cohorts eligible for retention analysis.

In [18]:
# Check retention observation window

query = """
WITH retention_clean AS (
    SELECT DISTINCT *
    FROM mercadolibre_retention
)

SELECT
    MIN(activity_date) AS first_activity_date,
    MAX(activity_date) AS last_activity_date,
    MIN(day_after_signup) AS min_day_after_signup,
    MAX(day_after_signup) AS max_day_after_signup,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT signup_date) AS signup_days
FROM retention_clean;
"""

retention_window = pd.read_sql_query(query, connection)

display(retention_window)

,first_activity_date,last_activity_date,min_day_after_signup,max_day_after_signup,unique_users,signup_days
0,2025-01-01,2025-09-22,0,28,12000,237


## 3. Data Quality Summary

The initial exploration covered two SQLite tables: `mercadolibre_funnel` and `mercadolibre_retention`.

### Key Findings

| Area | Finding | Decision |
|---|---|---|
| Funnel dataset | 61,235 events from 2,901 unique users | Use the original event data for funnel analysis, with appropriate date filtering. |
| Retention dataset | 696,000 rows from 12,000 unique users | Use distinct records for retention calculations. |
| Duplicate records | 6 exact duplicates in Funnel; 348,000 in Retention | Review Funnel duplicates before removing them. Every distinct Retention record appears twice, so exclude duplicate copies from user-day retention calculations. |
| Missing values | No NULL or empty values in the key identifiers and fields checked | No corrections required for these fields. |
| Retention dates | No inconsistencies found between signup dates, activity dates, and days since signup | Use the existing date fields. |
| Funnel timestamps | `event_times` has limited precision and differs from `event_date` | Use standardized `event_date` for chronological analysis, while documenting its limitations. |
| Date formatting | 25,684 event dates initially failed SQLite parsing | Standardizing single-digit hours resolved all parsing failures. |
| Retention observation window | Activity is recorded from D0 to D28, ending September 22, 2025 | Verify user-level observation coverage when calculating retention. |

### Data Preparation Decisions

- Keep the original SQLite database unchanged.
- Apply data corrections within SQL queries.
- Use `SELECT DISTINCT` to prevent duplicate retention records from affecting calculations.
- Standardize event date formatting before constructing chronological funnels.
- Define consistent analysis periods before calculating conversion and retention rates.

### Next Steps

The analysis will continue with:

1. **Conversion Funnel Analysis:** Measure user progression through the purchase journey and identify the largest observed drop-offs.
2. **Geographic Segmentation:** Compare funnel conversion rates across countries.
3. **Cohort Retention Analysis:** Calculate D7, D14, D21, and D28 retention rates.
4. **Business Recommendations:** Translate the findings into actionable opportunities.